In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import torch
import pickle as pickle
import matplotlib.pyplot as plt
import pandas as pd
import time
from veloproj import *
import unitvelo as utv
import os.path
from os.path import exists
import time
import dynamo as dyn 
scv.settings.verbosity = 3
method = 'veloAE'

(Running UniTVelo 0.2.5.2)
2024-12-16 11:43:11


In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

In [ ]:
for dataset in datasets:
    print(dataset)
    parser = get_parser()
    args = parser.parse_args(args=['--lr',  '1e-6',
                                '--n-epochs', '20000', 
                                '--g-rep-dim', '100',
                                '--k-dim', '100',
                                '--model-name', 'pancreas_scv_model.cpt',
                                '--exp-name', 'CohAE_pancreas_scv',
                                '--device', 'cuda:2',
                                '--gumbsoft_tau', '1',
                                '--nb_g_src', 'X',
                                '--ld_nb_g_src', "X",
                                '--n_raw_gene', '2000',
                                '--n_conn_nb', '30',
                                '--n_nb_newadata', '30',
                                '--aux_weight', '1',
                                '--fit_offset_train', 'false',
                                '--fit_offset_pred', 'true',
                                '--use_offset_pred', 'false',
                                '--gnn_layer', 'GAT',
                                '--vis-key', 'X_umap',
                                '--vis_type_col', 'clusters',
                                '--scv_n_jobs', '10',
                                ])
    args   
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed(args.seed)
    np.random.seed(args.seed)
    torch.backends.cudnn.deterministic = True

    device = torch.device(args.device if args.device.startswith('cuda') and torch.cuda.is_available() else "cpu")
    exp_metrics = {}
    def main_AE(args, adata):
        spliced = adata.layers['Ms']
        unspliced = adata.layers['Mu']
        tensor_s = torch.FloatTensor(spliced).to(device)
        tensor_u = torch.FloatTensor(unspliced).to(device)
        tensor_x = torch.FloatTensor(adata.X.toarray()).to(device)
        tensor_v = torch.FloatTensor(adata.layers['stc_velocity']).to(device)

        model = init_model(adata, args, device)

        inputs = [tensor_s, tensor_u]
        xyids = [0, 1]
        if args.use_x:
            inputs.append(tensor_x)

        model = fit_model(args, adata, model, inputs, tensor_v, xyids, device)
        
        return tensor_s, tensor_u, tensor_x 
    adata = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
    
    start = time.time() 
    scv.pp.neighbors(adata, n_neighbors=30, n_pcs=30)
    scv.utils.show_proportions(adata)
    scv.pp.filter_and_normalize(adata, min_shared_counts=30, n_top_genes=args.n_raw_gene)
    scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
    adata.obs.clusters.unique()
    file = open(data_dir + 'groundTruth.pickle' ,'rb')
    cluster_edges = pickle.load(file)
    scv.tl.velocity(adata, vkey='stc_velocity', mode="stochastic")
    scv.tl.velocity_graph(adata, vkey='stc_velocity', n_jobs=args.scv_n_jobs)
    scv.tl.velocity_confidence(adata, vkey='stc_velocity')
    scv.pl.velocity_embedding_stream(adata,  # legend_loc="right margin", 
                                    vkey="stc_velocity", basis=args.vis_key, color=args.vis_type_col,
                                    dpi=150, 
                                    title='ScVelo Stochastic Mode')
    exp_metrics["stc_mode"] = evaluate(adata, cluster_edges, args.vis_type_col, "stc_velocity")
    tensor_s, tensor_u, tensor_x = main_AE(args, adata)
    model = init_model(adata, args, device)
    model.load_state_dict(torch.load(args.model_name))
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        x = model.encoder(tensor_x)
        s = model.encoder(tensor_s)
        u = model.encoder(tensor_u)
        
        v = estimate_ld_velocity(s, u, device=device, perc=[5, 95], 
                                 norm=args.use_norm, fit_offset=args.fit_offset_pred, 
                                 use_offset= not args.use_offset_pred).cpu().numpy()
        x = x.cpu().numpy()
        s = s.cpu().numpy()
        u = u.cpu().numpy()
    
    adata = new_adata(adata, x, s, u, v, g_basis=args.ld_nb_g_src, n_nb_newadata=args.n_nb_newadata)
    scv.tl.velocity_graph(adata, vkey='new_velocity', n_jobs=args.scv_n_jobs)
    end = time.time()

    fix, ax = plt.subplots(1, 1, figsize = (8, 6))
    scv.pl.velocity_embedding_stream(adata, vkey="new_velocity", basis=args.vis_key, color=args.vis_type_col,ax = ax)
    plt.savefig(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.svg')
    # Calculate performance metrics:
    file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
    ground_truth = pickle.load(file)
    metrics = utv.evaluate(adata, ground_truth, 'clusters', 'new_velocity')
    if exists(save_dir + method +'/'+ '_CBDir_scores.csv'):
        tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
    else:
        tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
        tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
    ##CBDC_scores
    cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
    tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
    df_CB = pd.concat([df_CB, pd.DataFrame([[np.mean(cb_score), end - start]], columns=df_CB.columns, index=[dataset])])
    tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
    ##ICCoh_scores
    IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                for x in metrics['In-cluster Coherence'].keys()]
    tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]
    df_IC = pd.concat([df_IC, pd.DataFrame([[np.mean(IC_score), end - start]], columns=df_IC.columns, index=[dataset])])
    tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
    
    adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [ ]:
print(df_CB)
print(df_IC)
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores.csv')